# DMET-SSTKG — Cholec80/M2CAI Remote-Update Ablation

This notebook compares two hub targets and two update objectives on the
same-ontology Cholec80/M2CAI remote cohort. One maximum-entropy motif is selected
from each phase-aligned fragment, and every configuration is evaluated with the
same selected queries, leave-one-case-out folds, update-head architecture, optimizer,
and training schedule.

The annotation of the held-out case is used only to calculate the evaluation
metrics.

## 1. Generated Result

The notebook produces two reproducibility artifacts:

- `Cholec80_M2CAI_RemoteAblation.csv`: micro and phase-balanced gains for
  the four target–objective combinations, with percentile 95% confidence
  intervals from 10,000 paired surgical-case cluster bootstrap resamples;
- `Cholec80_M2CAI_RemoteAblation_BootstrapProtocol.json`: the bootstrap
  unit, resample count, seed, confidence-interval method, and phase-completeness
  rule used to generate the reported intervals.

Every bootstrap draw samples complete surgical cases with replacement. All
selected query motifs belonging to a sampled case remain together, preserving
within-case dependence and the pairing between each hub-based method and its
matched sequence-label reference.


## 2. Required Environment

Install the dependencies before running the notebook:

```bash
pip install torch==2.4.1 torch-geometric pandas numpy scikit-learn tqdm
```

Select the PyTorch build appropriate for the local CPU or CUDA environment.
The retained configuration uses CPU execution.

In [ ]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

## 3. Repository Paths

Edit the paths in the next cell to match the local repository layout.

- `REMOTE_SSTKG_ROOT`: directory containing the remote cohort's `Case_*` folders;
- `REMOTE_CACHE_DIR`: binary cache created from the remote S-STKG files;
- `MOTIF_CHECKPOINT`: phase-initialized motif-extractor checkpoint;
- `HUB_FILE`: serialized Dirichlet-conformal source hub;
- `RESULTS_DIR`: directory for the ablation result file.

To reproduce the reported cohort, place its 10 retained cases under
`REMOTE_SSTKG_ROOT`. Case-folder identifiers are used to construct the
leave-one-case-out folds. The node-level `case_id` value is set to zero before
model input.

In [ ]:
REMOTE_SSTKG_ROOT = Path("data/cholec80_m2cai_remote_sstkg")
REMOTE_CACHE_DIR = Path("outputs/cholec80_m2cai_remote_cache")

MOTIF_CHECKPOINT = Path(
    "artifacts/best_phase_initialized_motif_extractor.pt"
)
HUB_FILE = Path(
    "artifacts/Shareable_Motif_Hub_DIRICHLET_CONFORMAL_PROTOTYPE.pt"
)

RESULTS_DIR = Path("outputs/cholec80_m2cai_remote_ablation")

FORCE_REPROCESS_CACHE = False

## 4. Ablation Design

The experiment changes two components while keeping the remaining evaluation
conditions fixed.

| Component | Evaluated settings |
|---|---|
| Hub target | Full Dirichlet-calibrated distribution; distribution restricted to the conformal candidate set |
| Update objective | Soft cross-entropy; conformal-weighted Dirichlet hub supervision |

For each configuration, the matched reference trains the same update head on the
remote sequence-head prediction. The hub-based configuration replaces this target
with the selected hub distribution. Under the conformal-weighted objective, broader
candidate sets receive lower weight and focal modulation uses `gamma=2`.

In [ ]:
SEED = 42
DEVICE = torch.device("cpu")
EPS = 1e-12

CHUNK_SIZE = 100
MIN_CHUNK_FRAMES = 10

NUM_MODEL_CLASSES = 8
NODE_FEATURE_DIM = 11
EDGE_FEATURE_DIM = 3
HIDDEN_DIM = 128
MOTIF_DIM = 128
MOTIF_WINDOW = 16
MOTIF_STRIDE = 8

REMOVE_CASE_ID_FEATURE = True
REMOVE_G_RATIO_FEATURE = False
REMOVE_P_RATIO_FEATURE = False

CASE_ID_COL = 7
G_RATIO_COL = 8
P_RATIO_COL = 9
VIRTUAL_FLAG_COL = 10

UPDATE_EPOCHS = 500
UPDATE_LEARNING_RATE = 2e-2
UPDATE_WEIGHT_DECAY = 1e-4
UPDATE_MIN_SAMPLE_WEIGHT = 0.05
UPDATE_CONFORMAL_WEIGHT_POWER = 0.5
UPDATE_FOCAL_GAMMA = 2.0

BOOTSTRAP_N = 10000
BOOTSTRAP_SEED = 42
BOOTSTRAP_MAX_ATTEMPT_MULTIPLIER = 50

SOURCE_PHASE_NAMES = {
    1: "Trocar Placement",
    2: "Preparation",
    3: "Calot Triangle Dissection",
    4: "Clipping and Cutting",
    5: "Gallbladder Dissection",
    6: "Gallbladder Packaging",
    7: "Cleaning and Coagulation",
    8: "Gallbladder Retraction",
}

COHORT_SPEC = {
    "dataset_name": "Cholec80/M2CAI",
    "expected_cases": 10,
    "label_mode": "source_original_1based",
    "native_raw_to_source_original_id": {},
    "active_source_original_ids": list(range(1, 9)),
}

## 5. Labels and Reproducibility

The same-ontology cohort uses the complete P1–P8 source phase space. The
following helpers fix the random seed, convert one-based source phase IDs to the
zero-based model indices, normalize probability distributions, and read the
fragment labels.

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def source_original_id_to_model_index(source_original_id):
    source_original_id = int(source_original_id)

    if not 1 <= source_original_id <= NUM_MODEL_CLASSES:
        raise ValueError(
            "Source phase ID must be in 1..8; "
            f"received {source_original_id}."
        )

    return source_original_id - 1


def model_index_to_source_original_id(model_index):
    model_index = int(model_index)

    if not 0 <= model_index < NUM_MODEL_CLASSES:
        raise ValueError(
            "Model phase index must be in 0..7; "
            f"received {model_index}."
        )

    return model_index + 1


def l2_normalize(values):
    values = np.asarray(values, dtype=np.float32)

    if values.size == 0:
        return values

    if values.ndim == 1:
        return values / (np.linalg.norm(values) + EPS)

    return values / (
        np.linalg.norm(values, axis=1, keepdims=True) + EPS
    )


def softmax_array(values, axis=-1):
    values = np.asarray(values, dtype=np.float64)
    values = values - np.max(values, axis=axis, keepdims=True)
    exponentials = np.exp(values)

    return exponentials / (
        exponentials.sum(axis=axis, keepdims=True) + EPS
    )


def normalize_active_probabilities(probabilities, active_model_indices):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    active_model_indices = np.asarray(active_model_indices, dtype=int)

    if probabilities.ndim == 1:
        output = np.zeros(NUM_MODEL_CLASSES, dtype=np.float64)
        output[active_model_indices] = np.clip(
            probabilities[active_model_indices],
            0.0,
            None,
        )
        total = output.sum()

        if total <= EPS:
            output[active_model_indices] = (
                1.0 / len(active_model_indices)
            )
        else:
            output /= total

        return output

    output = np.zeros(
        (probabilities.shape[0], NUM_MODEL_CLASSES),
        dtype=np.float64,
    )
    output[:, active_model_indices] = np.clip(
        probabilities[:, active_model_indices],
        0.0,
        None,
    )
    totals = output.sum(axis=1, keepdims=True)
    zero_rows = totals[:, 0] <= EPS
    output /= totals + EPS

    if np.any(zero_rows):
        output[zero_rows, :] = 0.0
        zero_indices = np.where(zero_rows)[0]
        output[
            np.ix_(
                zero_indices,
                active_model_indices,
            )
        ] = 1.0 / len(active_model_indices)

    return output


def normalized_entropy(probabilities, active_model_indices):
    active_probabilities = normalize_active_probabilities(
        probabilities,
        active_model_indices,
    )[np.asarray(active_model_indices, dtype=int)]
    active_probabilities = np.clip(
        active_probabilities,
        EPS,
        1.0,
    )

    return float(
        -(
            active_probabilities
            * np.log(active_probabilities)
        ).sum()
        / max(
            np.log(len(active_model_indices)),
            EPS,
        )
    )


def read_single_fragment_label(label_path):
    label_path = Path(label_path)
    label_table = pd.read_csv(
        label_path,
        sep=None,
        engine="python",
    )
    label_table.columns = [
        str(column).strip()
        for column in label_table.columns
    ]

    if "phase_id" in label_table.columns:
        label_column = "phase_id"
    elif len(label_table.columns) >= 2:
        label_column = label_table.columns[1]
    else:
        raise ValueError(
            "A phase-label column could not be inferred from "
            f"{label_path}."
        )

    values = sorted(
        label_table[label_column]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if len(values) != 1:
        raise ValueError(
            "Each phase-aligned fragment must contain exactly "
            f"one phase label; found {values} in {label_path}."
        )

    return int(values[0])


def map_fragment_label_to_model_index(raw_label, cohort_spec):
    raw_label = int(raw_label)
    label_mode = str(cohort_spec["label_mode"])

    if label_mode == "source_original_1based":
        source_original_id = raw_label
    elif label_mode == "source_model_0based":
        source_original_id = raw_label + 1
    elif label_mode == "native_raw_0based":
        mapping = {
            int(key): int(value)
            for key, value
            in cohort_spec["native_raw_to_source_original_id"].items()
        }

        if raw_label not in mapping:
            raise ValueError(
                f"Native phase label {raw_label} is not defined "
                f"for {cohort_spec['dataset_name']}."
            )

        source_original_id = mapping[raw_label]
    else:
        raise ValueError(
            f"Unsupported label mode: {label_mode}"
        )

    active_ids = {
        int(value)
        for value in cohort_spec["active_source_original_ids"]
    }

    if source_original_id not in active_ids:
        raise ValueError(
            f"Mapped source phase P{source_original_id} is outside "
            f"the active phase space for {cohort_spec['dataset_name']}."
        )

    return (
        source_original_id_to_model_index(source_original_id),
        source_original_id,
    )


def one_hot_targets(model_indices):
    model_indices = np.asarray(model_indices, dtype=int)
    targets = np.zeros(
        (len(model_indices), NUM_MODEL_CLASSES),
        dtype=np.float64,
    )
    targets[
        np.arange(len(model_indices)),
        model_indices,
    ] = 1.0
    return targets


## 6. Remote S-STKG Cache

Each phase-aligned S-STKG segment is divided into ordered cache fragments with
at most 100 sampled frames. This matches the temporal input protocol used by the
source phase classifier and motif extractor.

A final residual fragment is retained only when it contains at least 10 sampled
frames, because shorter residuals provide very limited temporal context. The
node-level `case_id` value is set to zero before model input because it is
bookkeeping metadata rather than a surgical feature. The case-folder identifier
is retained only for the leave-one-case-out evaluation.

In [ ]:
class RemoteSSTKGDataset(Dataset):
    def __init__(
        self,
        root_dir,
        cache_dir,
        cohort_spec,
        chunk_size=100,
        minimum_chunk_frames=10,
        force_reprocess=False,
    ):
        super().__init__(str(root_dir))

        self.root_dir = Path(root_dir)
        self.cache_dir = Path(cache_dir)
        self.cohort_spec = dict(cohort_spec)
        self.chunk_size = int(chunk_size)
        self.minimum_chunk_frames = int(
            minimum_chunk_frames
        )

        if (
            force_reprocess
            and self.cache_dir.exists()
        ):
            shutil.rmtree(self.cache_dir)

        if (
            not self.cache_dir.exists()
            or not any(self.cache_dir.glob("*.pt"))
        ):
            self.cache_dir.mkdir(
                parents=True,
                exist_ok=True,
            )
            self._convert_and_fragment()
        else:
            print(
                "[OK] Existing remote S-STKG cache is reused: "
                f"{self.cache_dir}"
            )

        self.files = sorted(
            self.cache_dir.glob("*.pt")
        )

        if not self.files:
            raise RuntimeError(
                f"Remote binary cache is empty: {self.cache_dir}"
            )

        self._validate_cached_cases()

    @staticmethod
    def _required_paths(
        case_directory,
        fragment_prefix,
    ):
        case_directory = Path(case_directory)

        return {
            "nodes": (
                case_directory
                / f"{fragment_prefix}_nodes.txt"
            ),
            "edges": (
                case_directory
                / f"{fragment_prefix}_edges.txt"
            ),
            "edge_features": (
                case_directory
                / f"{fragment_prefix}_edge_features.txt"
            ),
            "labels": (
                case_directory
                / f"{fragment_prefix}_labels.txt"
            ),
        }

    @staticmethod
    def _frame_mask(
        table,
        frame_progress,
    ):
        return np.isclose(
            table.iloc[:, 0].to_numpy(
                dtype=float
            ),
            float(frame_progress),
            rtol=0.0,
            atol=1e-6,
        )

    def _clean_node_features(
        self,
        node_values,
    ):
        node_values = np.asarray(
            node_values,
            dtype=np.float32,
        ).copy()

        if REMOVE_CASE_ID_FEATURE:
            node_values[:, CASE_ID_COL] = 0.0

        if REMOVE_G_RATIO_FEATURE:
            node_values[:, G_RATIO_COL] = 0.0

        if REMOVE_P_RATIO_FEATURE:
            node_values[:, P_RATIO_COL] = 0.0

        return node_values

    def _discover_case_directories(self):
        if not self.root_dir.is_dir():
            raise FileNotFoundError(
                "Remote S-STKG root directory was not found: "
                f"{self.root_dir}"
            )

        case_directories = sorted(
            path
            for path in self.root_dir.iterdir()
            if (
                path.is_dir()
                and path.name.startswith("Case_")
            )
        )

        expected_cases = int(
            self.cohort_spec["expected_cases"]
        )

        if len(case_directories) != expected_cases:
            raise RuntimeError(
                f"{self.cohort_spec['dataset_name']} expects "
                f"{expected_cases} retained Case_* directories, "
                f"but {len(case_directories)} were found in "
                f"{self.root_dir}."
            )

        return case_directories

    def _convert_and_fragment(self):
        case_directories = (
            self._discover_case_directories()
        )

        fragment_index = 0
        skipped_incomplete = 0
        skipped_short = 0

        for case_directory in tqdm(
            case_directories,
            desc=(
                "Building "
                f"{self.cohort_spec['dataset_name']} "
                "remote cache"
            ),
        ):
            fragment_prefixes = sorted(
                {
                    path.name.replace(
                        "_nodes.txt",
                        "",
                    )
                    for path
                    in case_directory.glob("*_nodes.txt")
                }
            )

            for fragment_prefix in fragment_prefixes:
                paths = self._required_paths(
                    case_directory,
                    fragment_prefix,
                )

                missing_paths = [
                    path
                    for path in paths.values()
                    if not path.exists()
                ]

                if missing_paths:
                    skipped_incomplete += 1
                    continue

                node_table = pd.read_csv(
                    paths["nodes"]
                )
                edge_table = pd.read_csv(
                    paths["edges"]
                )
                edge_feature_table = pd.read_csv(
                    paths["edge_features"]
                )

                for table in [
                    node_table,
                    edge_table,
                    edge_feature_table,
                ]:
                    table.columns = [
                        str(column).strip()
                        for column in table.columns
                    ]

                expected_node_columns = [
                    "x",
                    "y",
                    "w",
                    "h",
                    "vx",
                    "vy",
                    "class",
                    "case_id",
                    "g_ratio",
                    "p_ratio",
                    "isV",
                ]

                if (
                    list(node_table.columns)
                    != expected_node_columns
                ):
                    raise ValueError(
                        "Unexpected node layout in "
                        f"{paths['nodes']}. "
                        f"Expected {expected_node_columns}; "
                        f"found {list(node_table.columns)}."
                    )

                raw_label = read_single_fragment_label(
                    paths["labels"]
                )
                (
                    true_model_index,
                    true_source_original_id,
                ) = map_fragment_label_to_model_index(
                    raw_label,
                    self.cohort_spec,
                )

                frame_progress_values = sorted(
                    node_table[
                        "g_ratio"
                    ].astype(float).unique()
                )

                for chunk_start in range(
                    0,
                    len(frame_progress_values),
                    self.chunk_size,
                ):
                    chunk_values = (
                        frame_progress_values[
                            chunk_start:
                            chunk_start
                            + self.chunk_size
                        ]
                    )

                    if (
                        len(chunk_values)
                        < self.minimum_chunk_frames
                    ):
                        skipped_short += 1
                        continue

                    graph_sequence = []

                    for frame_progress in chunk_values:
                        node_mask = np.isclose(
                            node_table[
                                "g_ratio"
                            ].to_numpy(
                                dtype=float
                            ),
                            float(frame_progress),
                            rtol=0.0,
                            atol=1e-6,
                        )

                        frame_nodes = (
                            node_table.loc[
                                node_mask
                            ]
                            .to_numpy(
                                dtype=np.float32,
                                copy=True,
                            )
                        )
                        frame_nodes = (
                            self._clean_node_features(
                                frame_nodes
                            )
                        )

                        if frame_nodes.shape[0] == 0:
                            raise RuntimeError(
                                "A sampled frame contains no nodes "
                                f"in {fragment_prefix}."
                            )

                        edge_mask = self._frame_mask(
                            edge_table,
                            frame_progress,
                        )
                        edge_feature_mask = self._frame_mask(
                            edge_feature_table,
                            frame_progress,
                        )

                        frame_edges = (
                            edge_table.loc[
                                edge_mask
                            ].to_numpy()
                        )
                        frame_edge_features = (
                            edge_feature_table.loc[
                                edge_feature_mask
                            ].to_numpy()
                        )

                        if (
                            len(frame_edges)
                            != len(frame_edge_features)
                        ):
                            raise RuntimeError(
                                "Edge and edge-feature rows are "
                                "misaligned for "
                                f"{fragment_prefix} at "
                                f"g_ratio={frame_progress}."
                            )

                        if len(frame_edges) > 0:
                            edge_index = torch.tensor(
                                frame_edges[
                                    :,
                                    2:4,
                                ].astype(np.int64),
                                dtype=torch.long,
                            ).t().contiguous()
                        else:
                            edge_index = torch.zeros(
                                (2, 0),
                                dtype=torch.long,
                            )

                        if len(frame_edge_features) > 0:
                            edge_attr = torch.tensor(
                                frame_edge_features[
                                    :,
                                    4:7,
                                ].astype(np.float32),
                                dtype=torch.float,
                            )
                        else:
                            edge_attr = torch.zeros(
                                (
                                    0,
                                    EDGE_FEATURE_DIM,
                                ),
                                dtype=torch.float,
                            )

                        graph = Data(
                            x=torch.tensor(
                                frame_nodes,
                                dtype=torch.float,
                            ),
                            edge_index=edge_index,
                            edge_attr=edge_attr,
                            y=torch.tensor(
                                [true_model_index],
                                dtype=torch.long,
                            ),
                        )
                        graph.is_v = (
                            graph.x[
                                :,
                                VIRTUAL_FLAG_COL,
                            ]
                            == 1
                        )

                        if int(
                            graph.is_v.sum().item()
                        ) != 1:
                            raise RuntimeError(
                                "Each sampled frame must contain "
                                "exactly one virtual-context node."
                            )

                        graph_sequence.append(graph)

                    cache_path = (
                        self.cache_dir
                        / (
                            "fragment_"
                            f"{fragment_index:06d}"
                            ".pt"
                        )
                    )
                    torch.save(
                        {
                            "graph_seq": graph_sequence,
                            "metadata": {
                                "case_id": (
                                    case_directory.name
                                ),
                                "prefix": (
                                    fragment_prefix
                                ),
                                "raw_label": int(
                                    raw_label
                                ),
                                "true_model_index": int(
                                    true_model_index
                                ),
                                "true_source_original_id": int(
                                    true_source_original_id
                                ),
                                "chunk_start_gr": float(
                                    chunk_values[0]
                                ),
                                "chunk_end_gr": float(
                                    chunk_values[-1]
                                ),
                            },
                        },
                        cache_path,
                    )
                    fragment_index += 1

        print(
            "[CACHE] fragments="
            f"{fragment_index}; "
            "skipped_incomplete="
            f"{skipped_incomplete}; "
            "skipped_short="
            f"{skipped_short}"
        )

    def _validate_cached_cases(self):
        case_ids = set()

        for cache_path in self.files:
            cache_object = torch.load(
                cache_path,
                map_location="cpu",
                weights_only=False,
            )
            metadata = dict(
                cache_object.get(
                    "metadata",
                    {},
                )
                or {}
            )
            case_id = str(
                metadata.get(
                    "case_id",
                    "UNKNOWN",
                )
            )
            case_ids.add(case_id)

        if "UNKNOWN" in case_ids:
            raise RuntimeError(
                "Case metadata is required for "
                "leave-one-case-out evaluation."
            )

        expected_cases = int(
            self.cohort_spec["expected_cases"]
        )

        if len(case_ids) != expected_cases:
            raise RuntimeError(
                "The binary cache contains "
                f"{len(case_ids)} cases; "
                f"{expected_cases} are required."
            )

        self.case_ids = sorted(case_ids)

    def len(self):
        return len(self.files)

    def get_with_metadata(
        self,
        index,
    ):
        cache_object = torch.load(
            self.files[int(index)],
            map_location="cpu",
            weights_only=False,
        )

        if (
            not isinstance(cache_object, dict)
            or "graph_seq"
            not in cache_object
        ):
            raise ValueError(
                "Remote cache objects must contain "
                "graph_seq and metadata."
            )

        metadata = dict(
            cache_object.get(
                "metadata",
                {},
            )
            or {}
        )
        metadata[
            "fragment_file"
        ] = self.files[
            int(index)
        ].name

        return (
            cache_object["graph_seq"],
            metadata,
        )

    def get(self, index):
        graph_sequence, _ = (
            self.get_with_metadata(index)
        )
        return graph_sequence

## 7. Frozen Source-Trained Motif Encoder

Remote graph sequences are encoded by the retained source-trained motif
extractor. The checkpoint also generated the source phase prototypes, so source
prototypes and remote queries occupy the same normalized 128-dimensional motif
space. No remote-specific encoder training is performed.

In [ ]:
class PhaseInitializedSSTKGMotifExtractor(nn.Module):
    def __init__(
        self,
        in_channels=11,
        edge_dim=3,
        hidden_dim=128,
        motif_dim=128,
        num_phases=8,
        motif_window=16,
        motif_stride=8,
    ):
        super().__init__()

        self.hidden_dim = int(hidden_dim)
        self.motif_window = int(motif_window)
        self.motif_stride = int(motif_stride)

        self.graph_conv_1 = GATv2Conv(
            in_channels,
            hidden_dim,
            edge_dim=edge_dim,
        )
        self.graph_conv_2 = GATv2Conv(
            hidden_dim,
            hidden_dim,
            edge_dim=edge_dim,
        )
        self.virtual_context_gate = nn.Linear(
            hidden_dim * 2,
            1,
        )
        self.temporal_encoder = nn.GRU(
            hidden_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.phase_head = nn.Linear(
            hidden_dim * 3,
            num_phases,
        )

        self.window_fusion = nn.Linear(
            hidden_dim * 2,
            hidden_dim,
        )
        self.motif_projection = nn.Sequential(
            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                motif_dim,
            ),
        )
        self.motif_phase_head = nn.Linear(
            motif_dim,
            num_phases,
        )

    def encode_frame_graphs(
        self,
        graph_sequence,
    ):
        frame_embeddings = []

        for graph in graph_sequence:
            node_embeddings = (
                self.graph_conv_1(
                    graph.x,
                    graph.edge_index,
                    graph.edge_attr,
                )
                .relu()
            )
            node_embeddings = (
                self.graph_conv_2(
                    node_embeddings,
                    graph.edge_index,
                    graph.edge_attr,
                )
                .relu()
            )

            mean_embedding = (
                node_embeddings.mean(
                    dim=0,
                    keepdim=True,
                )
            )
            virtual_embedding = (
                node_embeddings[
                    graph.is_v
                ]
            )

            if virtual_embedding.size(0) == 0:
                virtual_embedding = mean_embedding
            else:
                virtual_embedding = (
                    virtual_embedding.mean(
                        dim=0,
                        keepdim=True,
                    )
                )

            gate_weight = torch.sigmoid(
                self.virtual_context_gate(
                    torch.cat(
                        [
                            mean_embedding,
                            virtual_embedding,
                        ],
                        dim=-1,
                    )
                )
            )

            frame_embeddings.append(
                gate_weight
                * virtual_embedding
                + (
                    1.0
                    - gate_weight
                )
                * mean_embedding
            )

        if not frame_embeddings:
            raise RuntimeError(
                "The graph sequence is empty."
            )

        return (
            torch.stack(frame_embeddings)
            .squeeze(1)
        )

    def contextualize_frames(
        self,
        frame_embeddings,
    ):
        recurrent_output, hidden_state = (
            self.temporal_encoder(
                frame_embeddings.unsqueeze(0)
            )
        )
        recurrent_output = (
            recurrent_output.squeeze(0)
        )

        forward_output = (
            recurrent_output[
                :,
                :self.hidden_dim,
            ]
        )
        backward_output = (
            recurrent_output[
                :,
                self.hidden_dim:,
            ]
        )

        temporal_context = (
            0.5
            * (
                forward_output
                + backward_output
            )
        )
        contextualized_frames = (
            0.5
            * frame_embeddings
            + 0.5
            * temporal_context
        )

        final_recurrent_state = torch.cat(
            [
                hidden_state[0],
                hidden_state[1],
            ],
            dim=-1,
        )
        temporal_mean_skip = (
            frame_embeddings.mean(
                dim=0,
                keepdim=True,
            )
        )
        sequence_readout = torch.cat(
            [
                final_recurrent_state,
                temporal_mean_skip,
            ],
            dim=-1,
        )

        return (
            contextualized_frames,
            sequence_readout,
        )

    def fuse_window(
        self,
        window_embeddings,
    ):
        return self.window_fusion(
            torch.cat(
                [
                    window_embeddings.mean(
                        dim=0,
                        keepdim=True,
                    ),
                    window_embeddings[
                        -1
                    ].unsqueeze(0),
                ],
                dim=-1,
            )
        )

    def extract_motif_signatures(
        self,
        contextualized_frames,
    ):
        sequence_length = int(
            contextualized_frames.shape[0]
        )

        if sequence_length < self.motif_window:
            fused_windows = self.fuse_window(
                contextualized_frames
            )
        else:
            fused_windows = torch.cat(
                [
                    self.fuse_window(
                        contextualized_frames[
                            start:
                            start
                            + self.motif_window
                        ]
                    )
                    for start in range(
                        0,
                        sequence_length
                        - self.motif_window
                        + 1,
                        self.motif_stride,
                    )
                ],
                dim=0,
            )

        return F.normalize(
            self.motif_projection(
                fused_windows
            ),
            p=2,
            dim=-1,
        )

    def forward(
        self,
        graph_sequence,
    ):
        frame_embeddings = (
            self.encode_frame_graphs(
                graph_sequence
            )
        )
        (
            contextualized_frames,
            sequence_readout,
        ) = self.contextualize_frames(
            frame_embeddings
        )
        motif_signatures = (
            self.extract_motif_signatures(
                contextualized_frames
            )
        )

        return {
            "motif_logits": (
                self.motif_phase_head(
                    motif_signatures
                )
            ),
            "motif_signatures": (
                motif_signatures
            ),
            "sequence_logits": (
                self.phase_head(
                    sequence_readout
                )
            ),
        }


def convert_development_checkpoint_keys(
    state_dict,
):
    prefix_map = {
        "conv1.": "graph_conv_1.",
        "conv2.": "graph_conv_2.",
        "context_attn.": (
            "virtual_context_gate."
        ),
        "rnn.": "temporal_encoder.",
        "fc.": "phase_head.",
        "window_fuse.": "window_fusion.",
        "motif_proj.": "motif_projection.",
        "motif_cls.": "motif_phase_head.",
    }

    converted = {}

    for key, value in state_dict.items():
        converted_key = key

        for old_prefix, new_prefix in prefix_map.items():
            if key.startswith(old_prefix):
                converted_key = (
                    new_prefix
                    + key[len(old_prefix):]
                )
                break

        converted[converted_key] = value

    return converted


def load_motif_extractor(
    checkpoint_path,
):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            "Motif-extractor checkpoint was not found: "
            f"{checkpoint_path}"
        )

    model = (
        PhaseInitializedSSTKGMotifExtractor(
            in_channels=NODE_FEATURE_DIM,
            edge_dim=EDGE_FEATURE_DIM,
            hidden_dim=HIDDEN_DIM,
            motif_dim=MOTIF_DIM,
            num_phases=NUM_MODEL_CLASSES,
            motif_window=MOTIF_WINDOW,
            motif_stride=MOTIF_STRIDE,
        )
        .to(DEVICE)
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )
    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )
    state_dict = (
        convert_development_checkpoint_keys(
            state_dict
        )
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )
    model.eval()

    return model

## 8. Dirichlet-Calibrated Conformal Hub

Each selected motif is compared with the normalized source phase prototypes.
The stored Dirichlet calibration parameters convert prototype similarities into
calibrated phase probabilities. The stored APS threshold `qhat` determines the
conformal candidate set and the corresponding restricted hub target.

In [ ]:
def prototype_probabilities(
    query_embeddings,
    prototype_embeddings,
):
    similarities = (
        l2_normalize(query_embeddings)
        @ l2_normalize(
            prototype_embeddings
        ).T
    )

    return softmax_array(
        similarities,
        axis=1,
    )


def apply_dirichlet_calibrator(
    query_embeddings,
    prototype_embeddings,
    calibrator,
):
    raw_probabilities = np.clip(
        prototype_probabilities(
            query_embeddings,
            prototype_embeddings,
        ),
        EPS,
        1.0,
    )
    log_probabilities = np.log(
        raw_probabilities
    )

    coefficients = np.asarray(
        calibrator["coef"],
        dtype=np.float64,
    )
    intercept = np.asarray(
        calibrator["intercept"],
        dtype=np.float64,
    )
    classes = np.asarray(
        calibrator["classes"],
        dtype=int,
    )

    calibrated_logits = (
        log_probabilities
        @ coefficients.T
        + intercept.reshape(1, -1)
    )
    observed_probabilities = (
        softmax_array(
            calibrated_logits,
            axis=1,
        )
    )

    probabilities = np.zeros(
        (
            len(query_embeddings),
            NUM_MODEL_CLASSES,
        ),
        dtype=np.float64,
    )

    for column_index, model_index in enumerate(
        classes
    ):
        probabilities[
            :,
            int(model_index),
        ] = observed_probabilities[
            :,
            column_index,
        ]

    return probabilities / (
        probabilities.sum(
            axis=1,
            keepdims=True,
        )
        + EPS
    )


def conformal_set_from_probabilities(
    probabilities,
    qhat,
    active_model_indices,
):
    candidate_sets = []
    candidate_sizes = []

    probabilities = (
        normalize_active_probabilities(
            probabilities,
            active_model_indices,
        )
    )

    for probability in probabilities:
        active_probability = probability[
            np.asarray(
                active_model_indices,
                dtype=int,
            )
        ]
        local_order = np.argsort(
            -active_probability
        )
        ordered_model_indices = np.asarray(
            active_model_indices,
            dtype=int,
        )[local_order]
        cumulative_probability = np.cumsum(
            probability[
                ordered_model_indices
            ]
        )
        crossing = np.where(
            cumulative_probability
            > float(qhat)
        )[0]

        if len(crossing) == 0:
            final_position = (
                len(ordered_model_indices)
                - 1
            )
        else:
            final_position = int(
                crossing[0]
            )

        candidate_set = (
            ordered_model_indices[
                :final_position + 1
            ]
            .astype(int)
            .tolist()
        )

        candidate_sets.append(
            candidate_set
        )
        candidate_sizes.append(
            len(candidate_set)
        )

    return (
        candidate_sets,
        np.asarray(
            candidate_sizes,
            dtype=int,
        ),
    )


def conformal_restricted_target(
    calibrated_probabilities,
    candidate_set,
    active_model_indices,
):
    calibrated_probabilities = (
        normalize_active_probabilities(
            calibrated_probabilities,
            active_model_indices,
        )
    )
    target = np.zeros(
        NUM_MODEL_CLASSES,
        dtype=np.float64,
    )

    valid_indices = [
        int(model_index)
        for model_index in candidate_set
        if int(model_index)
        in {
            int(value)
            for value in active_model_indices
        }
    ]

    if not valid_indices:
        return calibrated_probabilities

    target[valid_indices] = (
        calibrated_probabilities[
            valid_indices
        ]
    )
    total = target.sum()

    if total <= EPS:
        return calibrated_probabilities

    return target / total


def load_calibrated_hub(
    hub_path,
):
    hub_path = Path(hub_path)

    if not hub_path.exists():
        raise FileNotFoundError(
            f"Calibrated hub was not found: {hub_path}"
        )

    hub_object = torch.load(
        hub_path,
        map_location="cpu",
        weights_only=False,
    )

    required_top_level = {
        "embeddings",
        "records",
        "calibration",
    }
    missing_top_level = (
        required_top_level
        - set(hub_object)
    )

    if missing_top_level:
        raise RuntimeError(
            "Hub object is missing fields: "
            f"{sorted(missing_top_level)}"
        )

    prototype_embeddings = l2_normalize(
        np.asarray(
            hub_object["embeddings"],
            dtype=np.float32,
        )
    )
    records = pd.DataFrame(
        hub_object["records"]
    )

    if len(records) != len(
        prototype_embeddings
    ):
        raise RuntimeError(
            "Hub records and prototype rows "
            "have different lengths."
        )

    if "verified_phase_id" in records.columns:
        phase_ids = (
            records[
                "verified_phase_id"
            ]
            .astype(int)
            .to_numpy()
        )
        order = np.argsort(
            phase_ids
        )
        records = (
            records.iloc[order]
            .reset_index(drop=True)
        )
        prototype_embeddings = (
            prototype_embeddings[
                order
            ]
        )
        ordered_phase_ids = (
            records[
                "verified_phase_id"
            ]
            .astype(int)
            .tolist()
        )
    elif "phase_ids" in hub_object:
        ordered_phase_ids = [
            int(value)
            for value
            in hub_object["phase_ids"]
        ]
    else:
        ordered_phase_ids = list(
            range(
                len(
                    prototype_embeddings
                )
            )
        )

    if ordered_phase_ids != list(
        range(NUM_MODEL_CLASSES)
    ):
        raise RuntimeError(
            "Hub prototypes must be ordered "
            "by model phase indices 0..7."
        )

    if prototype_embeddings.shape != (
        NUM_MODEL_CLASSES,
        MOTIF_DIM,
    ):
        raise RuntimeError(
            "Unexpected hub prototype shape: "
            f"{prototype_embeddings.shape}; "
            "expected "
            f"({NUM_MODEL_CLASSES}, {MOTIF_DIM})."
        )

    calibration = dict(
        hub_object["calibration"]
    )
    dirichlet_calibrator = (
        calibration.get(
            "dirichlet_calibrator"
        )
    )
    conformal_calibrator = (
        calibration.get(
            "conformal_calibrator"
        )
    )

    if dirichlet_calibrator is None:
        raise RuntimeError(
            "Dirichlet calibrator is absent "
            "from the hub."
        )

    if conformal_calibrator is None:
        raise RuntimeError(
            "Conformal calibrator is absent "
            "from the hub."
        )

    classes = np.asarray(
        dirichlet_calibrator[
            "classes"
        ],
        dtype=int,
    )

    if (
        len(classes) == 0
        or classes.min() < 0
        or classes.max()
        >= NUM_MODEL_CLASSES
    ):
        raise RuntimeError(
            "Dirichlet class indices must "
            "lie in 0..7."
        )

    if "qhat" not in conformal_calibrator:
        raise RuntimeError(
            "The conformal threshold qhat "
            "is absent from the hub."
        )

    return {
        "object": hub_object,
        "records": records,
        "prototype_embeddings": (
            prototype_embeddings
        ),
        "dirichlet_calibrator": (
            dirichlet_calibrator
        ),
        "conformal_calibrator": (
            conformal_calibrator
        ),
    }

## 9. Maximum-Entropy Query Selection

The motif extractor may produce several motif signatures from one fragment. The
motif with the highest normalized motif-head entropy is selected because it is
the fragment's most uncertain local motif. Reference labels are not used during
this selection.

The hub provides both the full Dirichlet-calibrated distribution and the same
distribution restricted to the conformal candidate set. These are the two targets
compared in the ablation.

In [ ]:

def collect_uncertainty_selected_queries(
    model,
    dataset,
    hub,
    cohort_spec,
):
    active_source_original_ids = [
        int(value)
        for value in cohort_spec[
            "active_source_original_ids"
        ]
    ]
    active_model_indices = [
        source_original_id_to_model_index(value)
        for value in active_source_original_ids
    ]

    selected_rows = []
    selected_embeddings = []

    with torch.no_grad():
        for dataset_index in tqdm(
            range(len(dataset)),
            desc="Selecting maximum-entropy query motifs",
        ):
            graph_sequence_cpu, metadata = (
                dataset.get_with_metadata(dataset_index)
            )
            graph_sequence = [
                graph.to(DEVICE)
                for graph in graph_sequence_cpu
            ]
            model_output = model(graph_sequence)

            true_model_index = int(
                graph_sequence_cpu[0].y.item()
            )

            sequence_probabilities = (
                normalize_active_probabilities(
                    softmax_array(
                        model_output["sequence_logits"]
                        .detach()
                        .cpu()
                        .numpy()[0]
                    ),
                    active_model_indices,
                )
            )
            sequence_prediction = int(
                np.argmax(sequence_probabilities)
            )

            motif_probabilities = (
                normalize_active_probabilities(
                    softmax_array(
                        model_output["motif_logits"]
                        .detach()
                        .cpu()
                        .numpy(),
                        axis=1,
                    ),
                    active_model_indices,
                )
            )
            motif_signatures = l2_normalize(
                model_output["motif_signatures"]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            if len(motif_probabilities) != len(
                motif_signatures
            ):
                raise RuntimeError(
                    "Motif probability and signature counts do not match."
                )

            motif_entropies = np.asarray(
                [
                    normalized_entropy(
                        probability,
                        active_model_indices,
                    )
                    for probability in motif_probabilities
                ],
                dtype=np.float64,
            )
            query_index = int(np.argmax(motif_entropies))

            selected_signature = motif_signatures[
                query_index
            ]
            selected_hub_probability = (
                apply_dirichlet_calibrator(
                    selected_signature.reshape(1, -1),
                    hub["prototype_embeddings"],
                    hub["dirichlet_calibrator"],
                )[0]
            )
            selected_hub_probability = (
                normalize_active_probabilities(
                    selected_hub_probability,
                    active_model_indices,
                )
            )
            candidate_sets, candidate_sizes = (
                conformal_set_from_probabilities(
                    selected_hub_probability.reshape(1, -1),
                    hub["conformal_calibrator"]["qhat"],
                    active_model_indices,
                )
            )
            selected_candidate_set = candidate_sets[0]
            selected_candidate_size = int(
                candidate_sizes[0]
            )
            selected_hub_target = (
                conformal_restricted_target(
                    selected_hub_probability,
                    selected_candidate_set,
                    active_model_indices,
                )
            )

            selected_embeddings.append(selected_signature)
            selected_rows.append(
                {
                    "query_id": int(len(selected_rows)),
                    "dataset_index": int(dataset_index),
                    "case_id": str(metadata["case_id"]),
                    "prefix": str(metadata.get("prefix", "")),
                    "true_model_index": int(true_model_index),
                    "sequence_prediction": int(sequence_prediction),
                    "motif_entropy": float(
                        motif_entropies[query_index]
                    ),
                    "hub_conformal_set_size": int(
                        selected_candidate_size
                    ),
                    "hub_probabilities": (
                        selected_hub_probability
                    ),
                    "hub_conformal_target": (
                        selected_hub_target
                    ),
                }
            )

    selected_table = pd.DataFrame(selected_rows)
    selected_embedding_array = l2_normalize(
        np.asarray(
            selected_embeddings,
            dtype=np.float32,
        )
    )

    if len(selected_table) != len(dataset):
        raise RuntimeError(
            "Exactly one query motif must be selected per remote fragment."
        )

    return {
        "selected_queries": selected_table,
        "selected_embeddings": selected_embedding_array,
        "active_model_indices": active_model_indices,
    }


## 10. Leave-One-Case-Out Update

The evaluation is repeated once for every case. In each round, one case is held
out for testing and the selected queries from the remaining cases train the
linear update head. Feature standardization is estimated from the training cases
of that round. The held-out labels are used only to calculate correctness.

In [ ]:
class LinearRemoteUpdateHead(nn.Module):
    def __init__(
        self,
        input_dimension,
        number_of_classes=8,
    ):
        super().__init__()
        self.linear = nn.Linear(
            int(input_dimension),
            int(number_of_classes),
        )

    def forward(self, embeddings):
        return self.linear(embeddings)


def standardize_training_embeddings(
    training_embeddings,
):
    mean = training_embeddings.mean(
        axis=0,
        keepdims=True,
    ).astype(np.float32)
    standard_deviation = (
        training_embeddings.std(
            axis=0,
            keepdims=True,
        )
        .astype(np.float32)
    )
    standard_deviation[
        standard_deviation < 1e-6
    ] = 1.0

    return mean, standard_deviation


def apply_standardization(
    embeddings,
    mean,
    standard_deviation,
):
    return (
        (
            embeddings
            - mean
        )
        / standard_deviation
    ).astype(np.float32)


def normalize_targets_over_active_phases(
    targets,
    active_model_indices,
):
    targets = np.asarray(
        targets,
        dtype=np.float64,
    )
    output = np.zeros_like(
        targets,
        dtype=np.float64,
    )
    output[
        :,
        np.asarray(
            active_model_indices,
            dtype=int,
        ),
    ] = np.clip(
        targets[
            :,
            np.asarray(
                active_model_indices,
                dtype=int,
            ),
        ],
        0.0,
        None,
    )
    totals = output.sum(
        axis=1,
        keepdims=True,
    )

    if np.any(
        totals[:, 0] <= EPS
    ):
        raise ValueError(
            "At least one update target has "
            "zero mass over the active phases."
        )

    return output / (
        totals + EPS
    )


def train_remote_update_head(
    training_embeddings,
    training_targets,
    active_model_indices,
    focal_gamma,
    sample_weights,
):
    training_embeddings = np.asarray(
        training_embeddings,
        dtype=np.float32,
    )
    training_targets = (
        normalize_targets_over_active_phases(
            training_targets,
            active_model_indices,
        )
        .astype(np.float32)
    )
    sample_weights = np.asarray(
        sample_weights,
        dtype=np.float32,
    )

    if len(training_embeddings) < 2:
        raise RuntimeError(
            "At least two training queries are "
            "required for the update head."
        )

    mean, standard_deviation = (
        standardize_training_embeddings(
            training_embeddings
        )
    )
    standardized_embeddings = (
        apply_standardization(
            training_embeddings,
            mean,
            standard_deviation,
        )
    )

    set_seed(SEED)

    update_head = (
        LinearRemoteUpdateHead(
            input_dimension=(
                standardized_embeddings.shape[
                    1
                ]
            ),
            number_of_classes=(
                NUM_MODEL_CLASSES
            ),
        )
        .to(DEVICE)
    )
    optimizer = torch.optim.AdamW(
        update_head.parameters(),
        lr=UPDATE_LEARNING_RATE,
        weight_decay=(
            UPDATE_WEIGHT_DECAY
        ),
    )

    embedding_tensor = torch.tensor(
        standardized_embeddings,
        dtype=torch.float32,
        device=DEVICE,
    )
    target_tensor = torch.tensor(
        training_targets,
        dtype=torch.float32,
        device=DEVICE,
    )
    weight_tensor = torch.tensor(
        sample_weights,
        dtype=torch.float32,
        device=DEVICE,
    )
    last_loss = np.nan

    for _ in range(
        int(UPDATE_EPOCHS)
    ):
        optimizer.zero_grad(
            set_to_none=True
        )

        full_logits = update_head(
            embedding_tensor
        )
        full_probabilities = F.softmax(
            full_logits,
            dim=1,
        )
        full_log_probabilities = (
            F.log_softmax(
                full_logits,
                dim=1,
            )
        )

        cross_entropy = -(
            target_tensor
            * full_log_probabilities
        ).sum(dim=1)

        if float(focal_gamma) > 0:
            target_agreement = (
                target_tensor
                * full_probabilities
            ).sum(dim=1)
            focal_weight = torch.pow(
                (
                    1.0
                    - target_agreement
                ).clamp_min(0.0),
                float(focal_gamma),
            )
            loss_vector = (
                focal_weight
                * cross_entropy
            )
        else:
            loss_vector = (
                cross_entropy
            )

        normalized_weights = (
            weight_tensor
            / (
                weight_tensor.mean()
                + EPS
            )
        )
        loss = (
            loss_vector
            * normalized_weights
        ).mean()
        loss.backward()
        optimizer.step()

        last_loss = float(
            loss.detach()
            .cpu()
            .item()
        )

    update_head.eval()

    return {
        "head": update_head,
        "mean": mean,
        "standard_deviation": (
            standard_deviation
        ),
        "last_loss": last_loss,
    }


def predict_remote_update_head(
    trained_head,
    evaluation_embeddings,
    active_model_indices,
):
    standardized_embeddings = (
        apply_standardization(
            np.asarray(
                evaluation_embeddings,
                dtype=np.float32,
            ),
            trained_head["mean"],
            trained_head[
                "standard_deviation"
            ],
        )
    )

    with torch.no_grad():
        full_logits = (
            trained_head["head"](
                torch.tensor(
                    standardized_embeddings,
                    dtype=torch.float32,
                    device=DEVICE,
                )
            )
            .detach()
            .cpu()
            .numpy()
        )

    active_probabilities = softmax_array(
        full_logits[
            :,
            np.asarray(
                active_model_indices,
                dtype=int,
            ),
        ],
        axis=1,
    )
    full_probabilities = np.zeros(
        (
            len(
                evaluation_embeddings
            ),
            NUM_MODEL_CLASSES,
        ),
        dtype=np.float64,
    )
    full_probabilities[
        :,
        np.asarray(
            active_model_indices,
            dtype=int,
        ),
    ] = active_probabilities

    return (
        full_probabilities.argmax(
            axis=1
        ).astype(int),
        full_probabilities,
    )


def target_matrix(
    selected_queries,
    target_name,
):
    if target_name == "sequence_label":
        return one_hot_targets(
            selected_queries[
                "sequence_prediction"
            ].to_numpy(
                dtype=int
            )
        )

    if target_name == "hub_distribution":
        return np.stack(
            selected_queries[
                "hub_probabilities"
            ].to_list()
        )

    if (
        target_name
        == "conformal_restricted_hub_distribution"
    ):
        return np.stack(
            selected_queries[
                "hub_conformal_target"
            ].to_list()
        )

    raise ValueError(
        f"Unknown update target: {target_name}"
    )


def training_sample_weights(
    selected_queries,
    target_name,
    objective_name,
):
    if (
        objective_name
        == "soft_cross_entropy"
        or target_name
        == "sequence_label"
    ):
        return np.ones(
            len(selected_queries),
            dtype=np.float32,
        )

    candidate_sizes = np.maximum(
        selected_queries[
            "hub_conformal_set_size"
        ]
        .astype(float)
        .to_numpy(),
        1.0,
    )
    weights = np.power(
        1.0 / candidate_sizes,
        UPDATE_CONFORMAL_WEIGHT_POWER,
    )
    weights = np.clip(
        weights,
        UPDATE_MIN_SAMPLE_WEIGHT,
        1.0,
    )

    return weights.astype(
        np.float32
    )


def objective_gamma(
    objective_name,
):
    if (
        objective_name
        == "soft_cross_entropy"
    ):
        return 0.0

    if (
        objective_name
        == "conformal_weighted_dirichlet_supervision"
    ):
        return float(
            UPDATE_FOCAL_GAMMA
        )

    raise ValueError(
        f"Unknown update objective: "
        f"{objective_name}"
    )


def run_leave_one_case_out_method(
    selected_queries,
    selected_embeddings,
    active_model_indices,
    target_name,
    objective_name,
):
    selected_queries = (
        selected_queries.copy()
        .reset_index(drop=True)
    )
    selected_embeddings = np.asarray(
        selected_embeddings,
        dtype=np.float32,
    )

    all_targets = target_matrix(
        selected_queries,
        target_name,
    )
    all_weights = (
        training_sample_weights(
            selected_queries,
            target_name,
            objective_name,
        )
    )

    packet_rows = []
    cases = sorted(
        selected_queries[
            "case_id"
        ].astype(str).unique().tolist()
    )

    for held_out_case in cases:
        training_mask = (
            selected_queries[
                "case_id"
            ].astype(str)
            != held_out_case
        ).to_numpy()
        evaluation_mask = (
            selected_queries[
                "case_id"
            ].astype(str)
            == held_out_case
        ).to_numpy()

        if (
            training_mask.sum() == 0
            or evaluation_mask.sum() == 0
        ):
            raise RuntimeError(
                "Every leave-one-case-out fold "
                "requires non-empty training and "
                "evaluation records."
            )

        trained_head = (
            train_remote_update_head(
                training_embeddings=(
                    selected_embeddings[
                        training_mask
                    ]
                ),
                training_targets=(
                    all_targets[
                        training_mask
                    ]
                ),
                active_model_indices=(
                    active_model_indices
                ),
                focal_gamma=(
                    objective_gamma(
                        objective_name
                    )
                ),
                sample_weights=(
                    all_weights[
                        training_mask
                    ]
                ),
            )
        )
        predictions, _ = (
            predict_remote_update_head(
                trained_head,
                selected_embeddings[
                    evaluation_mask
                ],
                active_model_indices,
            )
        )

        evaluation_rows = (
            selected_queries.loc[
                evaluation_mask
            ]
            .reset_index(drop=True)
        )

        for row_index, row in (
            evaluation_rows.iterrows()
        ):
            packet_rows.append(
                {
                    "query_id": int(
                        row["query_id"]
                    ),
                    "case_id": str(
                        row["case_id"]
                    ),
                    "true_model_index": int(
                        row[
                            "true_model_index"
                        ]
                    ),
                    "target_name": (
                        target_name
                    ),
                    "objective_name": (
                        objective_name
                    ),
                    "prediction": int(
                        predictions[
                            row_index
                        ]
                    ),
                    "correct": int(
                        predictions[
                            row_index
                        ]
                        == int(
                            row[
                                "true_model_index"
                            ]
                        )
                    ),
                }
            )

    packet_table = pd.DataFrame(
        packet_rows
    )

    if len(packet_table) != len(
        selected_queries
    ):
        raise RuntimeError(
            "Every selected query must be "
            "evaluated exactly once."
        )

    return packet_table


def run_required_update_methods(
    selected_queries,
    selected_embeddings,
    active_model_indices,
    include_ablation,
):
    method_definitions = [
        (
            "sequence_label",
            "conformal_weighted_dirichlet_supervision",
        ),
        (
            "conformal_restricted_hub_distribution",
            "conformal_weighted_dirichlet_supervision",
        ),
    ]

    if include_ablation:
        method_definitions.extend(
            [
                (
                    "sequence_label",
                    "soft_cross_entropy",
                ),
                (
                    "hub_distribution",
                    "soft_cross_entropy",
                ),
                (
                    "conformal_restricted_hub_distribution",
                    "soft_cross_entropy",
                ),
                (
                    "hub_distribution",
                    "conformal_weighted_dirichlet_supervision",
                ),
            ]
        )

    unique_definitions = []

    for definition in method_definitions:
        if definition not in unique_definitions:
            unique_definitions.append(
                definition
            )

    packet_tables = []

    for (
        target_name,
        objective_name,
    ) in unique_definitions:
        packet_tables.append(
            run_leave_one_case_out_method(
                selected_queries=(
                    selected_queries
                ),
                selected_embeddings=(
                    selected_embeddings
                ),
                active_model_indices=(
                    active_model_indices
                ),
                target_name=(
                    target_name
                ),
                objective_name=(
                    objective_name
                ),
            )
        )

    return pd.concat(
        packet_tables,
        ignore_index=True,
    )


## 11. Ablation Metrics

Micro gain is the pooled accuracy difference between a hub-based update and
its matched sequence-label reference over all selected query motifs.
Phase-balanced gain is calculated separately for each active surgical phase
and then averaged so that frequent phases do not dominate the result.

The point estimates use all selected queries. Confidence intervals use 10,000
paired surgical-case cluster bootstrap resamples. Surgical cases, rather than
individual motifs, are sampled with replacement; whenever a case is sampled,
all of its selected query motifs are retained together. This preserves
within-case dependence and method–baseline pairing. A phase-balanced bootstrap
draw is excluded only when it omits an active phase and therefore cannot
reproduce the predefined phase-balanced estimand.


In [ ]:
def method_packet(
    packet_table,
    target_name,
    objective_name,
):
    return (
        packet_table[
            (
                packet_table[
                    "target_name"
                ]
                == target_name
            )
            & (
                packet_table[
                    "objective_name"
                ]
                == objective_name
            )
        ]
        .copy()
        .sort_values(
            "query_id"
        )
        .reset_index(drop=True)
    )


def align_paired_packets(
    packet_table,
    method_target,
    method_objective,
    baseline_target,
    baseline_objective,
):
    method = method_packet(
        packet_table,
        method_target,
        method_objective,
    )[
        [
            "query_id",
            "case_id",
            "true_model_index",
            "correct",
        ]
    ].rename(
        columns={
            "correct": "method_correct"
        }
    )
    baseline = method_packet(
        packet_table,
        baseline_target,
        baseline_objective,
    )[
        [
            "query_id",
            "correct",
        ]
    ].rename(
        columns={
            "correct": "baseline_correct"
        }
    )

    paired = method.merge(
        baseline,
        on="query_id",
        how="inner",
        validate="one_to_one",
    )

    if len(paired) != len(method):
        raise RuntimeError(
            "Paired method and baseline packets "
            "do not contain the same queries."
        )

    if paired["case_id"].nunique() < 2:
        raise RuntimeError(
            "Surgical-case bootstrap requires "
            "at least two distinct cases."
        )

    return paired


def validate_paired_gain_frame(
    paired,
):
    required_columns = {
        "query_id",
        "case_id",
        "true_model_index",
        "method_correct",
        "baseline_correct",
    }
    missing_columns = (
        required_columns.difference(
            paired.columns
        )
    )

    if missing_columns:
        raise ValueError(
            "Paired gain table is missing "
            "required columns: "
            + ", ".join(
                sorted(
                    missing_columns
                )
            )
        )

    paired = (
        paired.copy()
        .dropna(
            subset=[
                "query_id",
                "case_id",
                "true_model_index",
                "method_correct",
                "baseline_correct",
            ]
        )
        .reset_index(drop=True)
    )
    paired["case_id"] = (
        paired["case_id"].astype(str)
    )
    paired["true_model_index"] = (
        paired[
            "true_model_index"
        ].astype(int)
    )
    paired["method_correct"] = (
        paired[
            "method_correct"
        ].astype(float)
    )
    paired["baseline_correct"] = (
        paired[
            "baseline_correct"
        ].astype(float)
    )

    if len(paired) == 0:
        raise ValueError(
            "Paired gain table contains "
            "no complete observations."
        )

    if (
        paired[
            "case_id"
        ].nunique()
        < 2
    ):
        raise ValueError(
            "Surgical-case bootstrap requires "
            "at least two cases."
        )

    return paired


def paired_difference(
    paired,
):
    return (
        paired[
            "method_correct"
        ].to_numpy(
            dtype=float
        )
        - paired[
            "baseline_correct"
        ].to_numpy(
            dtype=float
        )
    )


def case_cluster_components(
    paired,
):
    work = paired[
        [
            "case_id",
            "method_correct",
            "baseline_correct",
        ]
    ].copy()
    work["paired_difference"] = (
        work[
            "method_correct"
        ].astype(float)
        - work[
            "baseline_correct"
        ].astype(float)
    )

    grouped = (
        work.groupby(
            "case_id",
            sort=True,
        )[
            "paired_difference"
        ]
        .agg(
            [
                "sum",
                "count",
            ]
        )
        .reset_index()
    )

    return (
        grouped[
            "case_id"
        ].astype(str).to_numpy(),
        grouped[
            "sum"
        ].to_numpy(dtype=float),
        grouped[
            "count"
        ].to_numpy(dtype=float),
    )


def case_phase_cluster_components(
    paired,
    active_model_indices,
):
    case_ids = sorted(
        paired[
            "case_id"
        ].astype(str).unique().tolist()
    )
    active_model_indices = [
        int(index)
        for index
        in active_model_indices
    ]

    case_to_position = {
        case_id: position
        for position, case_id
        in enumerate(case_ids)
    }
    phase_to_position = {
        model_index: position
        for position, model_index
        in enumerate(
            active_model_indices
        )
    }

    difference_sums = np.zeros(
        (
            len(case_ids),
            len(active_model_indices),
        ),
        dtype=float,
    )
    observation_counts = np.zeros_like(
        difference_sums
    )

    work = paired[
        [
            "case_id",
            "true_model_index",
            "method_correct",
            "baseline_correct",
        ]
    ].copy()
    work["paired_difference"] = (
        work[
            "method_correct"
        ].astype(float)
        - work[
            "baseline_correct"
        ].astype(float)
    )

    grouped = (
        work.groupby(
            [
                "case_id",
                "true_model_index",
            ],
            sort=True,
        )[
            "paired_difference"
        ]
        .agg(
            [
                "sum",
                "count",
            ]
        )
        .reset_index()
    )

    for _, row in grouped.iterrows():
        model_index = int(
            row[
                "true_model_index"
            ]
        )
        if (
            model_index
            not in phase_to_position
        ):
            continue

        case_position = (
            case_to_position[
                str(
                    row[
                        "case_id"
                    ]
                )
            ]
        )
        phase_position = (
            phase_to_position[
                model_index
            ]
        )
        difference_sums[
            case_position,
            phase_position,
        ] = float(
            row[
                "sum"
            ]
        )
        observation_counts[
            case_position,
            phase_position,
        ] = float(
            row[
                "count"
            ]
        )

    return (
        np.asarray(
            case_ids,
            dtype=str,
        ),
        difference_sums,
        observation_counts,
    )


def draw_case_multiplicities(
    random_generator,
    number_of_draws,
    number_of_cases,
):
    probabilities = np.full(
        int(number_of_cases),
        1.0
        / float(
            number_of_cases
        ),
        dtype=float,
    )

    return (
        random_generator.multinomial(
            int(number_of_cases),
            probabilities,
            size=int(
                number_of_draws
            ),
        )
        .astype(float)
    )


def percentile_confidence_interval(
    bootstrap_values,
):
    bootstrap_values = np.asarray(
        bootstrap_values,
        dtype=float,
    )
    bootstrap_values = (
        bootstrap_values[
            np.isfinite(
                bootstrap_values
            )
        ]
    )

    if len(
        bootstrap_values
    ) == 0:
        raise RuntimeError(
            "No finite bootstrap values "
            "were generated."
        )

    lower, upper = np.percentile(
        bootstrap_values,
        [
            2.5,
            97.5,
        ],
    )

    return (
        float(lower),
        float(upper),
    )


def bootstrap_micro_gain(
    paired,
):
    paired = (
        validate_paired_gain_frame(
            paired
        )
    )
    observed = (
        100.0
        * float(
            paired_difference(
                paired
            ).mean()
        )
    )

    (
        _,
        case_difference_sums,
        case_observation_counts,
    ) = case_cluster_components(
        paired
    )

    random_generator = (
        np.random.default_rng(
            BOOTSTRAP_SEED
        )
    )
    case_multiplicities = (
        draw_case_multiplicities(
            random_generator,
            BOOTSTRAP_N,
            len(
                case_observation_counts
            ),
        )
    )

    bootstrap_numerators = (
        case_multiplicities
        @ case_difference_sums
    )
    bootstrap_denominators = (
        case_multiplicities
        @ case_observation_counts
    )
    bootstrap_values = (
        100.0
        * bootstrap_numerators
        / bootstrap_denominators
    )

    lower, upper = (
        percentile_confidence_interval(
            bootstrap_values
        )
    )

    return (
        observed,
        lower,
        upper,
    )


def observed_phase_balanced_gain(
    paired,
    active_model_indices,
):
    phase_gains = []

    for model_index in (
        active_model_indices
    ):
        phase_rows = paired[
            paired[
                "true_model_index"
            ].astype(int)
            == int(
                model_index
            )
        ]

        if len(
            phase_rows
        ) == 0:
            raise RuntimeError(
                "The observed evaluation "
                "does not contain active phase "
                f"index {model_index}."
            )

        phase_gains.append(
            float(
                paired_difference(
                    phase_rows
                ).mean()
            )
        )

    return (
        100.0
        * float(
            np.mean(
                phase_gains
            )
        )
    )


def phase_balanced_gain_and_ci(
    paired,
    active_model_indices,
):
    paired = (
        validate_paired_gain_frame(
            paired
        )
    )
    active_model_indices = [
        int(model_index)
        for model_index
        in active_model_indices
    ]

    observed = (
        observed_phase_balanced_gain(
            paired,
            active_model_indices,
        )
    )

    (
        _,
        case_phase_difference_sums,
        case_phase_observation_counts,
    ) = case_phase_cluster_components(
        paired,
        active_model_indices,
    )

    random_generator = (
        np.random.default_rng(
            BOOTSTRAP_SEED
        )
    )
    accepted_values = []
    attempted_draws = 0
    maximum_attempts = (
        int(
            BOOTSTRAP_N
        )
        * int(
            BOOTSTRAP_MAX_ATTEMPT_MULTIPLIER
        )
    )

    while (
        len(
            accepted_values
        )
        < int(
            BOOTSTRAP_N
        )
        and attempted_draws
        < maximum_attempts
    ):
        remaining_draws = (
            int(
                BOOTSTRAP_N
            )
            - len(
                accepted_values
            )
        )
        batch_size = min(
            max(
                remaining_draws * 2,
                1024,
            ),
            maximum_attempts
            - attempted_draws,
        )

        case_multiplicities = (
            draw_case_multiplicities(
                random_generator,
                batch_size,
                case_phase_difference_sums.shape[
                    0
                ],
            )
        )
        attempted_draws += (
            batch_size
        )

        phase_numerators = (
            case_multiplicities
            @ case_phase_difference_sums
        )
        phase_denominators = (
            case_multiplicities
            @ case_phase_observation_counts
        )
        complete_draws = np.all(
            phase_denominators
            > 0,
            axis=1,
        )

        if not np.any(
            complete_draws
        ):
            continue

        phase_gains = (
            phase_numerators[
                complete_draws
            ]
            / phase_denominators[
                complete_draws
            ]
        )
        draw_values = (
            100.0
            * phase_gains.mean(
                axis=1
            )
        )
        accepted_values.extend(
            draw_values.tolist()
        )

    if (
        len(
            accepted_values
        )
        < int(
            BOOTSTRAP_N
        )
    ):
        raise RuntimeError(
            "Could not obtain the requested "
            "number of complete surgical-case "
            "bootstrap draws while retaining "
            "all active phases. "
            f"Accepted {len(accepted_values)} "
            f"of {attempted_draws} attempts."
        )

    bootstrap_values = np.asarray(
        accepted_values[
            : int(
                BOOTSTRAP_N
            )
        ],
        dtype=float,
    )
    lower, upper = (
        percentile_confidence_interval(
            bootstrap_values
        )
    )

    return (
        observed,
        lower,
        upper,
    )


def build_ablation_table(
    packet_table,
    active_model_indices,
):
    rows = []

    target_labels = {
        "hub_distribution": (
            "Dirichlet-calibrated hub distribution"
        ),
        "conformal_restricted_hub_distribution": (
            "Conformal-set-restricted hub distribution"
        ),
    }
    objective_labels = {
        "soft_cross_entropy": (
            "Soft CE"
        ),
        "conformal_weighted_dirichlet_supervision": (
            "Conformal-weighted Dirichlet hub supervision"
        ),
    }

    for target_name in [
        "hub_distribution",
        "conformal_restricted_hub_distribution",
    ]:
        for objective_name in [
            "soft_cross_entropy",
            "conformal_weighted_dirichlet_supervision",
        ]:
            paired = align_paired_packets(
                packet_table=packet_table,
                method_target=(
                    target_name
                ),
                method_objective=(
                    objective_name
                ),
                baseline_target=(
                    "sequence_label"
                ),
                baseline_objective=(
                    objective_name
                ),
            )

            (
                micro_gain,
                micro_lower,
                micro_upper,
            ) = bootstrap_micro_gain(
                paired
            )
            (
                phase_gain,
                phase_lower,
                phase_upper,
            ) = phase_balanced_gain_and_ci(
                paired,
                active_model_indices,
            )

            rows.append(
                {
                    "hub_target": (
                        target_labels[
                            target_name
                        ]
                    ),
                    "training_objective": (
                        objective_labels[
                            objective_name
                        ]
                    ),
                    "num_cases": int(
                        paired[
                            "case_id"
                        ].nunique()
                    ),
                    "num_eval_motifs": int(
                        len(
                            paired
                        )
                    ),
                    "num_phases": int(
                        len(
                            active_model_indices
                        )
                    ),
                    "micro_gain_pp": float(
                        micro_gain
                    ),
                    "micro_ci95_low": float(
                        micro_lower
                    ),
                    "micro_ci95_high": float(
                        micro_upper
                    ),
                    "phase_balanced_gain_pp": float(
                        phase_gain
                    ),
                    "phase_ci95_low": float(
                        phase_lower
                    ),
                    "phase_ci95_high": float(
                        phase_upper
                    ),
                    "bootstrap_unit": (
                        "surgical_case_cluster"
                    ),
                    "bootstrap_resamples": int(
                        BOOTSTRAP_N
                    ),
                    "bootstrap_seed": int(
                        BOOTSTRAP_SEED
                    ),
                    "ci_method": (
                        "percentile_95"
                    ),
                    "phase_bootstrap_rule": (
                        "discard_draws_missing_active_phase"
                    ),
                }
            )

    return pd.DataFrame(
        rows
    )


## 12. Run the Ablation

The final call builds or reuses the cache, loads the frozen motif checkpoint and
calibrated hub, selects one query motif per fragment, evaluates the four
configurations, and writes the aggregate ablation result.

In [ ]:
def run_remote_update_ablation(
    remote_sstkg_root,
    remote_cache_dir,
    motif_checkpoint,
    hub_file,
    results_dir,
    force_reprocess_cache=False,
):
    set_seed(SEED)

    required_paths = [
        Path(motif_checkpoint),
        Path(hub_file),
    ]
    missing_paths = [
        path for path in required_paths
        if not path.exists()
    ]
    if missing_paths:
        raise FileNotFoundError(
            "Required model or hub artifacts are missing: "
            f"{missing_paths}"
        )

    dataset = RemoteSSTKGDataset(
        root_dir=remote_sstkg_root,
        cache_dir=remote_cache_dir,
        cohort_spec=COHORT_SPEC,
        chunk_size=CHUNK_SIZE,
        minimum_chunk_frames=MIN_CHUNK_FRAMES,
        force_reprocess=force_reprocess_cache,
    )
    model = load_motif_extractor(motif_checkpoint)
    hub = load_calibrated_hub(hub_file)

    extraction = collect_uncertainty_selected_queries(
        model=model,
        dataset=dataset,
        hub=hub,
        cohort_spec=COHORT_SPEC,
    )

    packet_table = run_required_update_methods(
        selected_queries=extraction["selected_queries"],
        selected_embeddings=extraction["selected_embeddings"],
        active_model_indices=extraction[
            "active_model_indices"
        ],
        include_ablation=True,
    )
    ablation_table = build_ablation_table(
        packet_table=packet_table,
        active_model_indices=extraction[
            "active_model_indices"
        ],
    )

    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    output_path = (
        results_dir
        / "Cholec80_M2CAI_RemoteAblation.csv"
    )
    protocol_path = (
        results_dir
        / "Cholec80_M2CAI_RemoteAblation_BootstrapProtocol.json"
    )
    ablation_table.to_csv(output_path, index=False)

    bootstrap_protocol = {
        "bootstrap_unit": "surgical_case_cluster",
        "paired_resampling": True,
        "resamples": int(BOOTSTRAP_N),
        "seed": int(BOOTSTRAP_SEED),
        "ci_method": "percentile_95",
        "confidence_level": 0.95,
        "micro_estimand": (
            "pooled motif-level accuracy gain with complete "
            "surgical cases resampled as clusters"
        ),
        "phase_balanced_estimand": (
            "mean of active-phase-specific accuracy gains with "
            "complete surgical cases resampled as clusters"
        ),
        "within_case_rule": (
            "all selected query motifs from each sampled case "
            "are retained together"
        ),
        "phase_completeness_rule": (
            "bootstrap draws omitting an active phase are discarded"
        ),
        "maximum_attempt_multiplier": int(
            BOOTSTRAP_MAX_ATTEMPT_MULTIPLIER
        ),
    }
    protocol_path.write_text(
        json.dumps(
            bootstrap_protocol,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\n[BOOTSTRAP PROTOCOL]")
    print(json.dumps(bootstrap_protocol, indent=2))
    print("\n[REMOTE-UPDATE ABLATION]")
    print(ablation_table.to_string(index=False))
    print(f"\nSaved: {output_path}")
    print(f"Saved: {protocol_path}")

    return {
        "dataset": dataset,
        "model": model,
        "hub": hub,
        "selected_queries": extraction[
            "selected_queries"
        ],
        "packet_table": packet_table,
        "ablation_table": ablation_table,
        "output_path": output_path,
        "bootstrap_protocol_path": protocol_path,
        "bootstrap_protocol": bootstrap_protocol,
    }


In [ ]:
ablation_run = run_remote_update_ablation(
    remote_sstkg_root=REMOTE_SSTKG_ROOT,
    remote_cache_dir=REMOTE_CACHE_DIR,
    motif_checkpoint=MOTIF_CHECKPOINT,
    hub_file=HUB_FILE,
    results_dir=RESULTS_DIR,
    force_reprocess_cache=FORCE_REPROCESS_CACHE,
)